# TP2b — Biogeochemistry: chlorophyll, nitrate and oxygen

**Suggested duration: ~1h30** · **Prerequisites**: TP2a (Copernicus Marine login, the `download()` helper), Course 1 and Course 2.

Course 1 said it: **biogeochemistry is the least observed part of the ocean**. Satellites see only the chlorophyll of the surface layer; nutrients and oxygen at depth come from a few ship samples, some BGC-Argo floats… and models. In this notebook you use both:
- **ocean colour**: satellite chlorophyll (an observation, surface only);
- **PISCES**: a biogeochemical model, which gives nitrate and oxygen everywhere, down to 1000 m.

Same region and same story as TP2a: the **Papagayo jet** in winter, the **Costa Rica Dome**, and now what they mean for life in the ocean.

## Learning objectives
By the end of this notebook you should be able to:
1. Download satellite chlorophyll and a biogeochemical model from Copernicus Marine.
2. Map a variable that spans several orders of magnitude, with a **logarithmic** colour scale.
3. Compare a seasonal cycle at two places.
4. Read vertical sections of **nitrate** and **oxygen**, and locate the **oxygen minimum zone**.
5. **Cross-check** a model against observations, and explain why they differ.

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete**. Everything else is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like.
- **🚀 Advanced track** blocks are optional: do them only if you finished the main exercise.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click.
- Stuck for more than 10 minutes? Ask a trainer — that's what we're here for.

> **⚠️ Before you start: work on your own copy.** In the JupyterLab file browser, right-click this notebook → *Duplicate*, then rename the copy to `02b_bgc_EN_mine.ipynb` and open it.

> ⚠️ **Download etiquette**: keep the boxes and periods **as given**. Every file here is < 15 MB.

---
## Setup — the same as TP2a

Run this cell as is: the same study box, `download()`, `show_dataset()` and `base_map()` as in TP2a. Your files go to the same `data_tp2/` folder, next to the TP2a files (keep it for Thursday).

In [ ]:
import shutil
from datetime import datetime, timezone
from pathlib import Path

import copernicusmarine
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

# Study area: Pacific coast of Costa Rica (Gulf of Papagayo + Costa Rica Dome)
LON_MIN, LON_MAX = -92, -80
LAT_MIN, LAT_MAX = 4, 14

# Where your downloads go (created next to this notebook)
DATA_DIR = Path("data_tp2")
DATA_DIR.mkdir(exist_ok=True)

# Instructor's backup copy on Kabre, used only if a download fails
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"
BACKUP_DIR = Path(OBS_DIR) / "TP2_BACKUP"


def download(fname, **kwargs):
    # Download a Copernicus Marine subset to DATA_DIR/fname, skip if it already exists
    todo = [k for k, v in kwargs.items() if v is ...]
    if todo:
        raise ValueError(f"Complete the TODOs first: {todo}")
    out = DATA_DIR / fname
    if out.exists():
        print(f"✓ {out} already there, download skipped ({out.stat().st_size / 1e6:.1f} MB)")
        return out
    try:
        copernicusmarine.subset(output_directory=str(DATA_DIR), output_filename=fname, **kwargs)
    except Exception as err:
        backup = BACKUP_DIR / fname
        if not backup.exists():
            raise
        print(f"⚠ Download failed ({err}).\n  Using the instructor's backup copy: {backup}")
        shutil.copy(backup, out)
    print(f"✓ {out} ({out.stat().st_size / 1e6:.1f} MB)")
    return out


def show_dataset(dataset_id):
    # Print variables, units and coordinate ranges of a Copernicus Marine dataset
    cat = copernicusmarine.describe(dataset_id=dataset_id, disable_progress_bar=True)
    product = cat.products[0]
    part = product.datasets[0].versions[0].parts[0]
    service = next(s for s in part.services if s.service_name == "arco-geo-series")
    print(f"{product.title}\n  product : {product.product_id}\n  dataset : {dataset_id}\n")
    for var in service.variables:
        print(f"  variable {var.short_name!r:20} units: {var.units:10} ({var.standard_name})")
    print()
    for coord in service.variables[0].coordinates:
        if coord.coordinate_id == "time":
            t0, t1 = (datetime.fromtimestamp(v / 1000, tz=timezone.utc).date()
                      for v in (coord.minimum_value, coord.maximum_value))
            print(f"  time      : {t0} → {t1}")
        elif coord.values:
            print(f"  {coord.coordinate_id:9} : {len(coord.values)} levels, {coord.values[0]} → {coord.values[-1]}")
        else:
            print(f"  {coord.coordinate_id:9} : {coord.minimum_value:.3f} → {coord.maximum_value:.3f}, step {coord.step:.3f}")


def base_map(ax, title="", extent=None):
    # extent = [lon_min, lon_max, lat_min, lat_max], the Costa Rica box by default
    ax.set_extent(extent or [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#d4c9a8", zorder=2)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.4, linestyle="--", zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    if title:
        ax.set_title(title, fontsize=11)
    return ax


print(f"Study area: lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")
print(f"Downloads go to: {DATA_DIR.resolve()}")

from matplotlib.colors import LogNorm     # logarithmic colour scale (Exercise 1)

The two datasets of this notebook:

| Exercise | What | `dataset_id` | Variables | Resolution |
|---|---|---|---|---|
| 1 | Satellite chlorophyll, L4 (GlobColour, several sensors merged) | `cmems_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M` | `CHL` (mg m⁻³) | 4 km, monthly |
| 2 | Biogeochemical model PISCES (hindcast) | `cmems_mod_glo_bgc_my_0.25deg_P1M-m` | `chl` (mg m⁻³), `no3`, `o2` (mmol m⁻³) | 0.25°, monthly, 0–5700 m |

Note the **capital letters**: the satellite variable is `CHL`, the model one is `chl`. Variable names are case-sensitive.

---
## Exercise 1 — Satellite chlorophyll

Chlorophyll-a is the pigment of phytoplankton: its concentration is a proxy for the **biomass of phytoplankton**, the base of the marine food web. Satellites estimate it from the **colour of the sea** (more chlorophyll → greener water).

**🎯 Your turn**: download `CHL` over the study box, for the **whole year 2023**.

In [ ]:
# TODO: complete the arguments (see the table above)
CHL_FILE = download(
    "chl_globcolour_costarica_2023.nc",
    dataset_id=...,         # TODO
    variables=...,          # TODO (a list!)
    minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
    minimum_latitude=LAT_MIN, maximum_latitude=LAT_MAX,
    start_datetime=...,     # TODO
    end_datetime=...,       # TODO
)

ds_chl = xr.open_dataset(CHL_FILE)
print(ds_chl)
print("\nLatitudes increasing?", bool(ds_chl.latitude[0] < ds_chl.latitude[-1]))

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- First line of the table: the `dataset_id`, and `variables=["CHL"]` (capital letters).
- Dates: `"2023-01-01"` to `"2023-12-31"`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M",
variables=["CHL"],
start_datetime="2023-01-01",
end_datetime="2023-12-31",
```

</details>

**✅ Check**: a file of ~3.4 MB, `time: 12` (one map per month, dated on the 1st), `latitude: 240`, `longitude: 288` (1/24° ≈ 4 km), latitudes increasing.

### 1b — Why a logarithmic colour scale?

First, the January–March mean with an ordinary (linear) colour scale. Run the cell:

In [ ]:
chl = ds_chl["CHL"]
chl_jfm_linear = chl.sel(time=slice("2023-01", "2023-03")).mean(dim="time")

fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": ccrs.PlateCarree()})
chl_jfm_linear.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="viridis",
                    cbar_kwargs={"label": "Chlorophyll (mg m$^{-3}$)"})
base_map(ax, "Chlorophyll, Jan–Mar 2023 — linear scale")
plt.show()
print(f"min = {float(chl_jfm_linear.min()):.2f}   median = {float(chl_jfm_linear.median()):.2f}   max = {float(chl_jfm_linear.max()):.2f} mg m-3")

Almost everything is dark: a few coastal points reach ~18 mg m⁻³, while the open ocean is at ~0.1. Chlorophyll varies over **several orders of magnitude** (0.05 → 20), so we use a **logarithmic** scale: each colour step multiplies the value instead of adding to it. In matplotlib: `norm=LogNorm(vmin=0.05, vmax=5)`.

**🎯 Your turn**: compute the mean chlorophyll of **January–March** (the Papagayo wind season) and of **August–October**, then compare the two maps with the same log scale.

In [ ]:
# TODO: the two seasonal means (hint: .sel(time=slice(...)) then .mean(dim="time"))
chl_jfm = ...
chl_aso = ...

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, field, title in zip(axes, [chl_jfm, chl_aso], ["January–March 2023", "August–October 2023"]):
    im = field.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="viridis",
                    norm=LogNorm(vmin=0.05, vmax=5), add_colorbar=False)
    base_map(ax, f"Chlorophyll, {title}")
fig.colorbar(im, ax=axes, shrink=0.8, label="Chlorophyll (mg m$^{-3}$), log scale")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- January–March: `chl.sel(time=slice("2023-01", "2023-03")).mean(dim="time")`; same idea for August–October.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
chl_jfm = chl.sel(time=slice("2023-01", "2023-03")).mean(dim="time")
chl_aso = chl.sel(time=slice("2023-08", "2023-10")).mean(dim="time")
```

</details>

**✅ Check**:
- **January–March**: a band of high chlorophyll (**≈ 0.6–0.7 mg m⁻³**) spreads west-south-west from the Gulf of Papagayo over the Costa Rica Dome, exactly where TP2a showed the cold tongue under the wind jet. The open ocean is at ~0.1–0.2, the Caribbean side ~0.1. Values > 5 only right at the coast.
- **August–October**: the bloom is gone; values > 0.5 only along the coasts (4 % of the box, against 13 % in January–March).
- White patches = **no data**: clouds hide the sea (about 15 % of the pixels in January–March). This monthly L4 product is merged from several sensors but **not gap-filled** (Course 1: L3/L4).

💬 **Why** a bloom in January–March? The Papagayo wind brings cold, **nutrient-rich** water up to the sunlit layer (TP2a). Exercise 2 shows these nutrients.

### 1c — The seasonal cycle at two places

**🎯 Your turn**: extract the monthly chlorophyll at the Gulf of Papagayo (87.5°W, 10.5°N) and in the open ocean to the south-west (90°W, 5°N), and plot them on a log axis.

In [ ]:
points = {"Gulf of Papagayo (87.5°W, 10.5°N)": (-87.5, 10.5),
          "Open ocean (90°W, 5°N)": (-90.0, 5.0)}

fig, ax = plt.subplots(figsize=(10, 4))
for name, (lon0, lat0) in points.items():
    # TODO: chlorophyll time series at the grid point nearest to (lon0, lat0)
    ts = ...
    ts.plot(ax=ax, marker="o", label=name)
ax.set_yscale("log")
ax.set_ylabel("Chlorophyll (mg m$^{-3}$)")
ax.set_title("Monthly chlorophyll, 2023")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `chl.sel(longitude=lon0, latitude=lat0, method="nearest")`, as in TP1a and TP1b.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
ts = chl.sel(longitude=lon0, latitude=lat0, method="nearest")
```

</details>

**✅ Check**: at Papagayo, **≈ 0.6–0.7 mg m⁻³ from January to March**, then a drop to ~0.15 from April, with small rebounds in October and December (~0.3). In the open ocean, ~0.1 all year long, with no real season. On a log axis, Papagayo in winter is about **6 times** richer than the open ocean.

---
## Exercise 2 — Below the surface: nitrate and oxygen from a model

No satellite sees nitrate or oxygen at depth. **PISCES** is a biogeochemical model: it computes nutrients, plankton and oxygen, driven by the currents and temperature of an ocean reanalysis. This product is a **hindcast**: a model run over past years, with no assimilation of biogeochemical observations.

**🎯 Your turn**: download `chl`, `no3` and `o2` over the study box, for **2023**, **between 0 and 1000 m**.

In [ ]:
# TODO: complete the arguments
BGC_FILE = download(
    "pisces_costarica_2023.nc",
    dataset_id=...,        # TODO
    variables=...,         # TODO (3 variables, lower case)
    minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
    minimum_latitude=LAT_MIN, maximum_latitude=LAT_MAX,
    minimum_depth=...,     # TODO
    maximum_depth=...,     # TODO
    start_datetime="2023-01-01",
    end_datetime="2023-12-31",
)

ds_bgc = xr.open_dataset(BGC_FILE)
print(ds_bgc)
print("\nDepth levels (m):", ds_bgc.depth.values.round(1))

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Second line of the table: the `dataset_id`; `variables=["chl", "no3", "o2"]`.
- Depth: `minimum_depth=0, maximum_depth=1000`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1M-m",
variables=["chl", "no3", "o2"],
minimum_depth=0, maximum_depth=1000,
```

</details>

**✅ Check**: a file of ~13 MB, `time: 12`, `depth: 46` levels from 0.5 m to 947 m (closer together near the surface), `latitude: 41`, `longitude: 49` (0.25° grid: much coarser than the 4 km of the satellite). Units: mmol m⁻³ for `no3` and `o2` (1 mmol m⁻³ ≈ 1 µmol L⁻¹).

### 2b — Two sections along 9°N

**🎯 Your turn**: compute the **annual mean** of the dataset, then select the latitude closest to **9°N** (the same section as in TP1b and TP2a, across the Costa Rica Dome). The cell then draws nitrate and oxygen side by side.

On the oxygen section, two lines:
- **60 mmol m⁻³**: a common threshold for **hypoxia**, below which many fish and invertebrates are stressed;
- **20 mmol m⁻³**: the core of the **oxygen minimum zone** (OMZ).

In [ ]:
LAT_SECTION = 9

# TODO: annual mean of the whole dataset, then the latitude closest to LAT_SECTION
sec = ...

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
sec["no3"].plot.contourf(ax=axes[0], x="longitude", y="depth", levels=np.arange(0, 44, 2),
                         cmap="YlGnBu", cbar_kwargs={"label": "Nitrate (mmol m$^{-3}$)"})
sec["no3"].plot.contour(ax=axes[0], x="longitude", y="depth", levels=[20], colors="k", linewidths=1.5)
sec["o2"].plot.contourf(ax=axes[1], x="longitude", y="depth", levels=np.arange(0, 230, 10),
                        cmap="RdYlBu", cbar_kwargs={"label": "Oxygen (mmol m$^{-3}$)"})
cs = sec["o2"].plot.contour(ax=axes[1], x="longitude", y="depth", levels=[20, 60],
                            colors="k", linestyles=["solid", "dashed"], linewidths=1.5)
axes[1].clabel(cs, fmt="%d")
for ax in axes:
    ax.set_ylim(600, 0)                      # depth pointing down
    ax.set_facecolor("#d4c9a8")              # land
    ax.set_xlabel("Longitude (°E)")
axes[0].set_title(f"PISCES nitrate, {float(sec.latitude):.2f}°N, mean 2023 (line: 20 mmol m$^{{-3}}$)")
axes[1].set_title(f"PISCES oxygen, {float(sec.latitude):.2f}°N, mean 2023")
axes[0].set_ylabel("Depth (m)")
axes[1].set_ylabel("")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `.mean(dim="time")` on the whole Dataset, then `.sel(latitude=LAT_SECTION, method="nearest")`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
bgc_mean = ds_bgc.mean(dim="time")
sec = bgc_mean.sel(latitude=LAT_SECTION, method="nearest")
```

</details>

**✅ Check**:
- **Nitrate**: almost none at the surface (~1–2 mmol m⁻³: phytoplankton has used it up), then a sharp increase, the **nitracline**: the 20 mmol m⁻³ line is at ~55–60 m in the west and ~75 m near the coast. Below 150 m, ~40 mmol m⁻³. This deep reservoir is what the winter winds bring up to the surface.
- **Oxygen**: ~200 mmol m⁻³ in the surface layer, then a fast decrease. **Hypoxic** (< 60) from ~70 m at 92°W and ~110–120 m around 88–89°W; **< 20** from ~220–260 m offshore, rising to ~110 m near the coast; from ~370 m to the bottom of the section the water is almost **anoxic** (~1 mmol m⁻³). East of 84°W (the narrow Caribbean strip), no OMZ: oxygen stays high.

💬 **For biologists**: the eastern tropical Pacific has one of the strongest **oxygen minimum zones** of the world ocean. Oxygen-demanding species (tunas, billfish…) are squeezed into the upper ~100 m: this is known as **habitat compression**. Satellite chlorophyll tells you nothing about it: only in situ profiles and models do.

---
## Exercise 3 — Cross-check: model vs satellite chlorophyll

Both products give the **surface chlorophyll** over the same box, month by month. Do they agree?

**🎯 Your turn**: compute the **area-weighted box mean** of the satellite `CHL` and of the model `chl` (first level, `isel(depth=0)`), for each month. The weights are given (`cos(latitude)`, as in TP1b).

In [ ]:
chl_model = ds_bgc["chl"].isel(depth=0)          # model surface level (0.5 m)

w_sat = np.cos(np.deg2rad(chl.latitude))
w_mod = np.cos(np.deg2rad(chl_model.latitude))

# TODO: area-weighted mean over latitude and longitude, for each month
box_sat = ...
box_mod = ...

table = pd.DataFrame({"satellite (mg m-3)": box_sat.values, "model (mg m-3)": box_mod.values},
                     index=box_sat.time.dt.strftime("%Y-%m").values)
table["model / satellite"] = table["model (mg m-3)"] / table["satellite (mg m-3)"]
print(table.round(2))
print(f"\nAnnual mean: satellite = {float(box_sat.mean()):.2f}   model = {float(box_mod.mean()):.2f} mg m-3")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `chl.weighted(w_sat).mean(["latitude", "longitude"])`; the same with `chl_model` and `w_mod`.
- Missing pixels (clouds) are simply ignored by `.weighted(...).mean(...)`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
box_sat = chl.weighted(w_sat).mean(["latitude", "longitude"])
box_mod = chl_model.weighted(w_mod).mean(["latitude", "longitude"])
```

</details>

**✅ Check**: satellite box mean between **≈ 0.27 and 0.49** mg m⁻³, model between **≈ 0.19 and 0.33**: annual means ≈ **0.33** (satellite) vs ≈ **0.22** (model). The model is about **one third lower**, but both peak in **February** and are lowest from May to September.

💬 **Discuss**: why do they differ, and which one is "right"? Think about:
- **resolution**: 4 km vs 25 km. The model cannot resolve the narrow coastal blooms (up to 18 mg m⁻³) that weigh in the satellite mean;
- **gaps**: the satellite mean ignores cloudy pixels, the model has none;
- **the satellite is not perfect** either: near the coast, sediments and dissolved matter can be mistaken for chlorophyll;
- **the model** has its own biases, and this hindcast assimilates no biogeochemical data.

Agreement on the **seasonal cycle** but not on the **absolute value** is a typical outcome of a model–observation comparison (Thursday!).

---
## 🐞 Debugging exercise

A colleague wants a log-scale map of the **annual mean model chlorophyll** at the surface. Their cell has **two bugs**: run it, read the **last line** of each error, fix, run again.

In [ ]:
# 🐞 Fix me! (2 bugs)
chl_year = ds_chl["chl"].mean(dim="time")
fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": ccrs.PlateCarree()})
chl_year.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="viridis", norm=LogNorm(vmin=0, vmax=5))
base_map(ax, "PISCES surface chlorophyll, mean 2023")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Bug 1: which dataset contains a variable called `chl` (lower case)? Print `list(ds_chl.data_vars)` and `list(ds_bgc.data_vars)`.
- Bug 2: what is log(0)?

</details>

**✅ Check**: the fixed cell draws a smooth, coarse (0.25°) map: higher chlorophyll along the Pacific coast (up to ~0.6 at the Gulf of Papagayo) and towards the Dome (~0.25), ~0.15–0.2 in the south of the box, and < 0.1 on the Caribbean side. Compare with the 4 km satellite maps of Exercise 1: same pattern, much less detail.

---
### 🚀 Advanced track — a map of the hypoxia depth

At each grid point, find the **shallowest depth where oxygen drops below 60 mmol m⁻³** (annual mean), and map it. Hint: `ds_bgc["o2"].mean(dim="time")`, then `bgc_o2.depth.where(bgc_o2 < 60).min(dim="depth")`: keep the depths where the condition is true, and take the smallest one.

Where is low-oxygen water closest to the surface? What happens on the Caribbean side?

In [ ]:
# 🚀 Your code here

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `o2_mean.depth.where(o2_mean < 60)` is NaN where the water is not hypoxic; `.min(dim="depth")` then gives the first hypoxic depth.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
o2_mean = ds_bgc["o2"].mean(dim="time")
hypoxia_depth = o2_mean.depth.where(o2_mean < 60).min(dim="depth")

fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": ccrs.PlateCarree()})
hypoxia_depth.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="magma_r", vmin=60, vmax=160,
                   cbar_kwargs={"label": "Depth where O$_2$ < 60 mmol m$^{-3}$ (m)"})
base_map(ax, "PISCES — top of the hypoxic layer, mean 2023")
plt.show()
```

</details>

**✅ Check (advanced)**: hypoxic water starts at only **~70–90 m** in a band running west from the Gulf of Papagayo along ~10°N, and along the coast; it is deeper (~100–130 m) elsewhere, and below 150 m in the south-west of the box. The **Caribbean side is blank**: oxygen never drops below 60 mmol m⁻³ there in the top 1000 m. Two oceans a few hundred kilometres apart, two very different habitats.

---
## Recap

| Dataset | Type | Resolution | Variables | Your file |
|---|---|---|---|---|
| GlobColour L4 | observation (satellite ocean colour, several sensors) | 4 km, monthly | `CHL` (surface only) | `data_tp2/chl_globcolour_costarica_2023.nc` |
| PISCES hindcast | biogeochemical model | 0.25°, monthly, 0–1000 m kept | `chl`, `no3`, `o2` | `data_tp2/pisces_costarica_2023.nc` |

- Chlorophyll spans orders of magnitude: map it with a **log scale**.
- The winter **Papagayo** wind → cold, **nitrate-rich** water at the surface → **chlorophyll bloom** (TP2a + this notebook).
- Below ~100 m, the eastern tropical Pacific is **hypoxic**, almost anoxic below ~400 m: invisible from space.
- Model and satellite agree on the **seasonal cycle**, not on the **absolute values**: keep this in mind for Thursday.